# 17 · Hybrid Search：稠密 + 稀疏 + 分数融合

> Dense 与 Sparse 分数**量纲不同，不能直接相加**。融合策略是混合检索的灵魂。

**本文件覆盖知识点**：Dense+Sparse / Score Fusion / Weighted Fusion / Reciprocal Rank Fusion(RRF) / Weighted RRF / Query Weight / Relative Score Fusion

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 33 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


## 1. 三类融合策略

| 策略 | 做法 | 特点 |
|------|------|------|
| **Score Fusion** | 把两边分数映射到同一量纲后相加 | 保留强度，需归一化（min-max/z-score） |
| **Weighted Fusion** | 加权求和：`w1·Sparse + w2·Dense` | 可调权重；仍要解决量纲 |
| **RRF** | 只看名次：`Σ 1/(k+rank)` | **无需统一量纲**，稳健，最常用 |

RRF 公式：`RRF(doc) = Σ over lists  1 / (k + rank_i(doc))`，`k` 常取 60。

> RRF 的精妙：它不在乎“BM25 给 12 分、向量给 0.83”这种不可比的分数，只看**各自榜单里的名次**。

In [ ]:
# 两个榜单用 RRF 融合（rrf_fuse 的公式与实现是本课教学点，原样不动；换成真实检索结果再喂进去）
def rrf_fuse(rankings, k=60):
    """rankings: 多个"文档id榜单"(从高到低)。返回融合分降序的 [(doc, score)]"""
    score = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            score[doc] = score.get(doc, 0) + 1.0 / (k + rank + 1)
    return sorted(score.items(), key=lambda x: -x[1])

QUERY = '星云客服机器人如何计费'
dense_list = dense_retrieve(QUERY, k=5)      # 真实向量榜（余弦，量纲 0~1）
bm25_list  = sparse_retrieve(QUERY, k=5)     # 真实 BM25 榜（词面分，量纲 0~30+，与余弦不可比）

def _lab(h):
    return '%s·%s' % (h['source'].replace('.md', ''), h['section'])

info = {}                                    # 文档id → 它在两榜里的名次，用来解释融合结果
for _name, _hits in (('dense', dense_list), ('bm25', bm25_list)):
    for _rank, _h in enumerate(_hits, 1):
        info.setdefault(_h['i'], {'lab': _lab(_h)})[_name] = _rank

print('查询：%s' % QUERY)
print('\n① 向量榜 dense（余弦相似度）')
for r, h in enumerate(dense_list, 1):
    print('   %d. %-30s %.4f  %s' % (r, _lab(h), h['score'], h['text'][:26].replace('\n', ' ')))
print('② BM25 榜 sparse（词面得分，注意量纲与上面完全不同）')
for r, h in enumerate(bm25_list, 1):
    print('   %d. %-30s %.4f  %s' % (r, _lab(h), h['score'], h['text'][:26].replace('\n', ' ')))

fused = rrf_fuse([[h['i'] for h in dense_list], [h['i'] for h in bm25_list]])
pos = {doc: r for r, (doc, _) in enumerate(fused, 1)}
print('\n③ RRF 融合后（k=60，只看名次，两榜因此可比）')
for r, (doc, s) in enumerate(fused[:5], 1):
    it = info[doc]
    print('   %d. %-30s %.6f   向量#%s / BM25#%s'
          % (r, it['lab'], s, it.get('dense', '—'), it.get('bm25', '—')))

# 观察由本次真实榜单算出来，不是写死的结论
count = lambda d: sum(1 for k in ('dense', 'bm25') if k in info[d])
both_top = [d for d, _ in fused[:5] if count(d) == 2]
singles = [(info[d]['lab'], '向量' if 'dense' in info[d] else 'BM25',
            info[d].get('dense') or info[d].get('bm25'), pos[d]) for d, _ in fused if count(d) == 1]
print('\n观察（由本次真实榜单算出）:')
print('  融合 Top-5 里 %d 条是两榜共同召回 —— RRF 分 = 两榜名次分之和，天然排在单榜命中的前面。' % len(both_top))
for lab, which, rank, p in singles[:3]:
    print('  · %-30s 只在 %s 榜(#%d)，仍被 RRF 留在融合榜第 %d 位' % (lab, which, rank, p))
print('\n→ RRF 不理会「BM25 给 31 分、向量给 0.77」这种不可比的原始分，只看名次：'
      '所以 BM25 独有召回的《故障排查》能被捞回融合榜，而两榜都靠前的《计费与 SLA 说明》稳居第一 ——'
      '这正是混合检索比单路召回稳的原因。')


## 2. 何时升级融合策略

- 起步直接用 **RRF**，几乎不用调参；
- 想要更优：**Weighted RRF / Relative Score Fusion**——给 Sparse、Dense 不同权重，或用每个列表内部相对的分数形态（如归一化到 [0,1]）加权；
- **Query Weight**：有些问题明显是“查精确型号”→ 临时调高 Sparse 权重。

融合后仍要再走 **Rerank（第22课）** 做最终精排，两段职责不同。



In [ ]:
# 知识点·真调说明：Query Weight —— 让模型把查询判成「精确词面 / 语义概念」，决定融合时该偏向 Sparse 还是 Dense
import json as _json
_fall = ('[{"query": "星云客服机器人 企业版 型号 SE-2200 的参数", "type": "exact", "reason": "型号是唯一词面标识"}, '
         '{"query": "客服机器人私有化部署需要满足哪些合规要求？", "type": "semantic", "reason": "合规要求需概念召回"}, '
         '{"query": "报错码 KB-503 是什么意思", "type": "exact", "reason": "报错码靠精确匹配"}, '
         '{"query": "客服机器人与人工客服相比，优势是什么？", "type": "semantic", "reason": "对比类需语义泛化"}]')
out = _llm_live(
    prompt="""判断下面 4 条用户查询在混合检索里更适合“关键词精确匹配（偏向 BM25 / Sparse）”还是“语义理解（偏向向量 / Dense）”，输出 JSON 数组，不要输出任何其它文字。

1. 星云客服机器人 企业版 型号 SE-2200 的参数
2. 客服机器人私有化部署需要满足哪些合规要求？
3. 报错码 KB-503 是什么意思
4. 客服机器人与人工客服相比，优势是什么？

JSON 格式：[{"query": 查询原文, "type": "exact" 或 "semantic", "reason": 一句话理由}]""",
    system='你是混合检索的查询分析器。exact=词面命中即有答案（型号/报错码/精确名称）；semantic=需要同义或概念泛化（比较/合规/优劣）。',
    fallback=_fall,
    temperature=0.2,
)
s = out if out is not None else _fall
s = s.strip().strip('`')
if s.startswith('json'):
    s = s[4:].strip()
if out is None:
    print('（以上为固定样例；下面用样例走 json.loads 解析）')
try:
    for it in _json.loads(s):
        if it['type'] == 'exact':
            tag = '→ 词面即答案：融合时该给 Sparse(BM25) 更大权重'
        else:
            tag = '→ 语义泛化：融合时该给 Dense(向量) 更大权重'
        print('• %s（%s）%s' % (it['query'], it['reason'], tag))
except Exception as e:
    print('未通过 json.loads：', e, '—— 说明结构化约束需再收紧')
print('→ 这就是 Query Weight 的输入：先判定查询类型，再临时调高 Sparse 或 Dense 的权重；RRF 本身无权重，可用 Weighted RRF 落地。')

## 3. 一个生产级检索的完整形态（预告）

```text
BM25 榜 ─┐
Dense 榜 ─┼─→ RRF 融合 → Top-50 候选池 → Rerank → Top-5 → LLM
Metadata ─┘   (前置过滤)
```

## 小结

- Hybrid = Dense + Sparse，量纲不同需**融合**；
- **RRF 按名次融合**，稳健零调参，是标配；追求更好再上 Weighted / 分数归一；
- 混合检索之后通常接 Rerank。